In [ ]:
import platform
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rc

#가속 연산 디바이스(GPU/MPS) 자동 설정
#CUDA -> MPS -> CPU 순으로 사용 가능한 자원 체크

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
print(f"현재 연산에 사용 중인 디바이스: {device}")

# 사전 학습된 kobert 감정 분류 모델 및 토크나이저 로드
model_name = "rkdaldus/ko-sent5-classification"
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

text_to_analyze = "TIL 작성이 너무 재밌고 신나서 한잔 했어"

encoded_inputs = tokenizer(text_to_analyze, return_tensors="pt", padding=True, truncation=True).to(device)

# 모델 추론을 통한 로짓(Logits) 값 추출
with torch.no_grad():
    model_outputs = model(**encoded_inputs)
    logits = model_outputs.logits

# Softmax 함수를 적용하여 감정별 확률 계산
calculated_probabilities = torch.nn.functional.softmax(logits, dim=-1).squeeze().tolist()

# 감정 레이블 매핑 및 데이터프레임 변환
emotion_labels = ["angry", "unrest", "happy", "tranquility", "sad"]
emotion_df = pd.DataFrame({
    "Emotion": emotion_labels,
    "Probability": calculated_probabilities
})

print("--- 분석 결과 ---")
print(emotion_df)

# 분석 결과 시각화 (막대그래프)
plt.figure(figsize=(8, 5))
bar_colors = ["#ff7f7f", "#ffbf7f", "#7fbf7f", "#7fbfbf", "#7f7fff"]
plt.bar(emotion_df["Emotion"], emotion_df["Probability"], color=bar_colors, edgecolor="black", width=0.5)

plt.title("KoBERT Student Dialogue Emotion Analysis", fontsize=13, fontweight="bold", pad=15)
plt.xlabel("Emotions", fontsize=11)
plt.ylabel("Confidence Probability (0.0 ~ 1.0)", fontsize=11)
plt.ylim(0, 1.1)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## 데이터베이스
- 데이터(Data) : 데이터는 사실이나 정보를 표현하는 원시 값(raw value)
    - 이름, 나이, 구매 금액, 날짜 등 의미가 없지만 맥락과 함께 분석되면 정보가 됨
- 데이터베이스(Database, DB) : 체계적으로 정리된 데이터의 집합
    - 단순히 쌓아두는것이 아닌 효율적으로 저장하고 빠르게 검색할 수 있도록 구조화된 저장 공간
- Query & SQL 
    - 쿼리는 데이터베이스에 보내는 질문 또는 명령
    - SQL은 쿼리를 작성하기 위한 표준화된 언어

- RDBMS : 관계형 데이터베이스로 행과 열로 구성된 테이블 형태의 구조를 가지고 있다. 유연성이 높고 SQL로 조회 가능하다
    - 왜 대부분 관계형을 쓸까?
        - 테이블 형태로 저장하여 직관적으로 이해하기 쉽고 여러 테이블을 연결하여 복잡한 데이터 관계를 유연하게 표현 가능
        - SQL 이라는 표준화된 언어를 사용하기 때문에 한번 배워두면 다양한 DBMS에서 동일하게 활용 가능
        - 무결성 보장, 트랜잭션 처리, 성능 최적화 등 기업에 필요한 기능이 잘 갖춰져 있음

- 테이블의 구성 요소
    - 테이블
        - 데이터의 집합
        - 회원 테이블, 주문 테이블
    - 행
        - 하나의 데이터 항목
        - 회원 한명의 정보
    - 열
        - 데이터의 속성
        - 이름, 이메일, 가입일
    - 셀
        - 행과 열이 만나는 최소 단위

- SQL 작성 순서와 실행 순서
    - 작성 순서
        - SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY
    - 실행 순서
        - FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY

- 사용 DBMS : PostgreSQL
- 사용 소프트웨어 : DBeaver